# Get to Know a Dataset: The BigBrain — AWS Open Data Version

This notebook provides a guided introduction to the **BigBrain** dataset for its entry in the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The BigBrain is an ultra-high-resolution 3D reconstruction of a complete human brain created from 7,404 coronal histological sections cut at 20 µm, stained for cell bodies, digitized at high resolution, and reconstructed in 3D. The broader BigBrain collection includes volumetric reconstructions, cortical surfaces, classified tissue volumes, cortical layer maps, cytoarchitectonic maps, selected high-resolution histology, and derived datasets.

> **AWS setup note:** The final S3 bucket name, AWS region, Registry landing-page URL, and any bucket-level dataset prefix will be inserted once the AWS Open Data deployment is finalized.


In [ ]:
# Required Python packages for this notebook:
#
# boto3
# nibabel
# nilearn
# matplotlib
# numpy
#
# Install them with pip or conda using the preferred method for your environment.


In [ ]:
from pathlib import Path
import tempfile

import boto3
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

from botocore import UNSIGNED
from botocore.config import Config
from nilearn import plotting


In [ ]:
# Final values to be inserted after the AWS Open Data bucket is provisioned.
bucket = "REPLACE_WITH_BIGBRAIN_S3_BUCKET"
region = "REPLACE_WITH_AWS_REGION"

# If the BigBrain collection lives at the bucket root, leave this as "".
# Otherwise set it to the final prefix, ending in "/".
dataset_prefix = "REPLACE_WITH_BIGBRAIN_DATASET_PREFIX/"

# Public bucket: do not sign requests.
s3 = boto3.client(
    "s3",
    region_name=None if region.startswith("REPLACE_") else region,
    config=Config(signature_version=UNSIGNED),
)

# List the first level of the BigBrain dataset in S3.
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="" if dataset_prefix.startswith("REPLACE_") else dataset_prefix,
    Delimiter="/",
    MaxKeys=100,
)

for prefix in response.get("CommonPrefixes", []):
    print("PREFIX:", prefix["Prefix"])

for obj in response.get("Contents", []):
    print("OBJECT:", obj["Key"], f"({obj['Size'] / (1024**2):.1f} MiB)")


Inspect the specific NIfTI directory used by the tested DataLad analysis. This performs metadata discovery only; it does not download the NIfTI files.


In [ ]:
# Use the same dataset-relative directory as in the prior validated DataLad-based notebook.
nifti_relative_prefix = "3D_Volumes/MNI-ICBM152_Space/nii/"
nifti_prefix = (
    nifti_relative_prefix
    if dataset_prefix.startswith("REPLACE_")
    else f"{dataset_prefix}{nifti_relative_prefix}"
)

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=nifti_prefix,
    MaxKeys=100,
)

nifti_objects = response.get("Contents", [])
for obj in nifti_objects:
    print(f"{obj['Key']}  ({obj['Size'] / (1024**2):.1f} MiB)")


Select the same 400 µm, 8-bit NIfTI reconstruction.

The code also queries the S3 object metadata before transfer so that the compressed object size can be confirmed.


In [ ]:
example_relative_path = Path(
    "3D_Volumes/MNI-ICBM152_Space/nii/full8_400um_2009b_sym.nii.gz"
)

file_key = (
    example_relative_path.as_posix()
    if dataset_prefix.startswith("REPLACE_")
    else f"{dataset_prefix}{example_relative_path.as_posix()}"
)

metadata = s3.head_object(Bucket=bucket, Key=file_key)
print("Selected S3 object:", file_key)
print(f"Compressed file size: {metadata['ContentLength'] / (1024**2):.1f} MiB")
print("Last modified:", metadata.get("LastModified"))
print("Content type:", metadata.get("ContentType"))


### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The BigBrain collection uses several established neuroscience and 3D geometry formats.

- **NIfTI (`.nii`, `.nii.gz`)** is widely used for volumetric neuroimaging. It stores voxel data together with spatial metadata describing how the image is positioned in physical space. Python users can work with NIfTI using packages such as `nibabel`, `nilearn`, `numpy`, and many neuroimaging toolkits.
- **MINC (`.mnc`)** is another volumetric medical-imaging format used extensively in the BigBrain and McGill neuroimaging ecosystem. MINC supports multidimensional image data and spatial metadata.
- **GIfTI (`.gii`)** is commonly used for neuroimaging surface geometry and associated surface data.
- **OBJ (`.obj`)** is a general 3D geometry format used for cortical and other surface meshes.

BigBrain is available in more than one format because researchers use a wide range of visualization, image-processing, neuroinformatics, and HPC environments. Where possible, the collection exposes equivalent representations so that users can work with the data using established tools.

For introductory Python workflows, a downsampled NIfTI volume is a convenient starting point because it can be loaded directly using `nibabel` and visualized with `nilearn`. For full-resolution data, users should avoid unnecessary transfers and instead work with an appropriate spatial subset, resolution, or compute resource close to the data.


### Q: Can you show us an example of downloading and loading data from your dataset?

The example below downloads the **400 µm BigBrain NIfTI volume**, but obtains it from the public AWS S3 bucket.

The S3 transfer is AWS-specific. After the file is available locally, the loading and analysis steps intentionally mirror prior validated notebooks.


In [ ]:
tmpdir = Path(tempfile.mkdtemp(prefix="bigbrain-aws-"))
local_file = tmpdir / example_relative_path.name

s3.download_file(bucket, file_key, str(local_file))

if not local_file.is_file():
    raise FileNotFoundError(f"S3 download did not make the selected file available: {local_file}")

print(f"Downloaded compressed file size: {local_file.stat().st_size / (1024 ** 2):.1f} MiB")

# Match the successfully tested DataLad variant: use memory mapping for the NIfTI file.
img = nib.load(local_file, mmap=True)

print("Loaded:", local_file.name)
print("Shape:", img.shape)
print("Voxel sizes (mm):", img.header.get_zooms()[:3])
print("Data type:", img.get_data_dtype())


The image header tells us the dimensions, voxel spacing, and data type without requiring us to manually interpret the file structure. These metadata are especially important for BigBrain because the dataset is distributed at multiple resolutions and in multiple spatial reference systems.


In [ ]:
print(img.header)


A useful next step is to inspect a spatial subset of the volume. The earlier BigBrain tutorial used `nibabel` slicing together with `nilearn` to explore a cropped portion of the reconstruction rather than rendering an entire large volume at once.


In [ ]:
# Select a central spatial crop.
# The crop is calculated from the actual image dimensions so it remains robust
# if a different downsampled NIfTI example is selected.

shape = np.array(img.shape[:3])
half_width = np.maximum(shape // 8, 1)
center = shape // 2

start = np.maximum(center - half_width, 0)
stop = np.minimum(center + half_width, shape)

cropped = img.slicer[
    start[0]:stop[0],
    start[1]:stop[1],
    start[2]:stop[2],
]

print("Original shape:", img.shape)
print("Cropped shape:", cropped.shape)


### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

A defining characteristic of BigBrain is that it connects whole-brain neuroanatomy with microscopic histology. Even a downsampled volume can be explored interactively, while higher-resolution representations support progressively finer inspection of cortical and subcortical structure.

The following visualization adapts the approach used in the earlier BigBrain tutorial: load a NIfTI volume with `nibabel`, select a spatial subset, and render it with `nilearn`.


In [ ]:
# Interactive slice viewer for the cropped BigBrain volume
view = plotting.view_img(
    cropped,
    bg_img=None,
    cmap="gray",
    resampling_interpolation="nearest",
)
view


For a static view, we can also display orthogonal cuts through the downsampled reconstruction.


In [ ]:
#plotting.plot_anat(
#    img,
#    title="BigBrain example volume",
#    display_mode="ortho",
#)
#plt.show()

# Extract every 5th voxel across all dimensions to reduce memory use by 125x
downsampled_data = img.dataobj[::5, ::5, ::5]

# Convert the downsampled slice back into a tiny Nifti image object
tiny_img = nib.Nifti1Image(downsampled_data, img.affine)

# Now it's small enough to run safely through nilearn without crashing
from nilearn import plotting
plotting.plot_img(tiny_img, display_mode="ortho", title="Downsampled BigBrain")
plt.savefig("downsampled_brain1.png")


In [ ]:
# Mathematically find the exact center voxel index
voxel_dims = np.array(tiny_img.shape)
center_voxel = voxel_dims // 2

# Transform the center voxel index into world coordinate space (mm)
affine = tiny_img.affine
math_cut_coordinates = affine[:3, :3] @ center_voxel + affine[:3, 3]
auto_cut_coordinates = tuple(math_cut_coordinates)

print(f"Dynamically calculated center coordinates: {auto_cut_coordinates}")

# Pass the calculated coordinates straight into your plot
plotting.plot_img(
    tiny_img, 
    display_mode="ortho", 
    cut_coords=auto_cut_coordinates, 
    title="Downsampled BigBrain"
)

# Save and view your image
plt.savefig("downsampled_brain2.png")
plt.show()

We can inspect the image intensity distribution as a simple way to understand the numerical range represented in the selected volume. This is not intended as a biological analysis; it is an introductory quality-control step that helps users become familiar with the image values before applying more specialized processing.


In [ ]:
data = np.asarray(cropped.dataobj)
finite_values = data[np.isfinite(data)]

print("Minimum:", finite_values.min())
print("Maximum:", finite_values.max())
print("Mean:", finite_values.mean())
print("Median:", np.median(finite_values))


The histogram below summarizes the voxel intensities in the selected spatial crop.


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(finite_values.ravel(), bins=100)
plt.title("Voxel intensity distribution in a BigBrain spatial crop")
plt.xlabel("Voxel intensity")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


A key practical lesson is that users do not always need the highest-resolution representation for every task. BigBrain is available at multiple resolutions and in multiple spaces so that researchers can choose an appropriate trade-off between anatomical detail, transfer size, memory use, and computational cost.


In [ ]:
voxel_sizes = np.array(img.header.get_zooms()[:3])
voxel_volume_mm3 = np.prod(voxel_sizes)
n_voxels = np.prod(img.shape[:3])

print(f"Voxel size: {voxel_sizes} mm")
print(f"Voxel volume: {voxel_volume_mm3:.6f} mm³")
print(f"Number of voxels: {n_voxels:,}")
print(f"Uncompressed voxel array size (approx.): {n_voxels * img.get_data_dtype().itemsize / (1024**3):.2f} GiB")


### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

> **Can we work with BigBrain data without first moving the entire dataset to a local workstation?**

The workflow above demonstrates the AWS version of selective retrieval:

1. inspect S3 prefixes and object metadata;
2. choose a lower-resolution representation appropriate for the task;
3. retrieve one selected object rather than the multi-terabyte collection;
4. memory-map the downloaded NIfTI with `nibabel`;
5. inspect its metadata and a local spatial crop; and
6. visualize and summarize the selected data using the same analytical operations that were successfully tested in the DataLad variant.

The intensity statistics and histogram describe the crop, including any background voxels. They are introductory numerical summaries, not tissue-specific biological findings. The memory estimate describes the uncompressed stored voxel array; scaling, temporary arrays, and visualization can require more RAM.


### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

One broad opportunity is to ask how newly developed image-analysis and machine-learning methods can take advantage of BigBrain's multiscale information while remaining computationally efficient and reproducible.

For example:

> **Can automated methods identify or characterize fine-grained cortical and cytoarchitectonic features across very large regions of the BigBrain while preserving anatomical context and remaining practical to execute at scale?**

A useful approach would be to:

1. begin with lower-resolution or spatially restricted BigBrain data for method development;
2. use existing cortical-layer and cytoarchitectonic maps as anatomical context or reference information where appropriate;
3. package analysis tools in containers;
4. describe command-line tools with portable interfaces such as Boutiques when suitable;
5. scale execution using HPC or cloud resources rather than relying on a single workstation; and
6. record software versions, parameters, spatial references, and provenance so results can be reproduced.

The AWS Open Data distribution is intended to make this type of scalable experimentation easier by placing an openly accessible copy of the BigBrain collection in object storage that can be reached directly from cloud and federated computational environments.

Additional BigBrain tools and tutorials are available through:
- https://bigbrainproject.org/tools-and-services.html
- https://siibra-python.readthedocs.io/
- https://bigbrainwarp.readthedocs.io/
- https://www.cbrain.ca/


# Before publishing

Before committing this AWS notebook to the GitHub repository:

1. Replace `REPLACE_WITH_BIGBRAIN_S3_BUCKET`, `REPLACE_WITH_AWS_REGION`, and `REPLACE_WITH_BIGBRAIN_DATASET_PREFIX/` with the final AWS values. If the dataset lives at the bucket root, set `dataset_prefix = ""`.
2. Confirm that the AWS object corresponding to `3D_Volumes/MNI-ICBM152_Space/nii/full8_400um_2009b_sym.nii.gz` exists at the expected key.
3. Restart the kernel and run all cells in order from a clean Python environment.
4. Confirm the S3 metadata queries, `head_object`, and public `download_file` calls work without AWS credentials.
5. Confirm that the NIfTI metadata, crop dimensions, interactive viewer, both static plots, intensity summaries, histogram, voxel volume, and memory calculation match the tested DataLad workflow.
6. Update the Registry landing-page link once the final BigBrain Registry slug is confirmed.
7. Clear outputs before committing the final version unless AWS specifically requests retained example outputs.